
# Workflow Overview: ND2 to OME-Zarr, Colony & Nucleus Segmentation, Feature Extraction

The following set of notebook provides a reproducible workflow for high-content image analysis of timelpase live cell experiments. The workflow is designed to process microscopy data from raw ND2 files to quantitative feature extraction, enabling downstream biological analysis. The main steps are:

1. **ND2 to OME-Zarr conversion**: Convert raw ND2 microscopy files to the OME-Zarr format for scalable, cloud-ready storage and analysis.
2. **Colony segmentation using ConvPaint**: Identify and segment stem cell colonies in the images using a deep learning-based approach.
3. **Nucleus segmentation using StarDist / Cellpose**: Detect and segment individual nuclei within colonies for single-cell analysis.
4. **Cell Tracking**: Track individual cells over time to study dynamic behaviors.
5. **Feature Extraction**: Quantify spatial features and extract relevant biological markers (e.g., ERK, Oct4) for each cell.

Configuration options such as the output path or scaling parameters can be easily adjusted in the `configuration/settings.py` file. To change the way the dask cluster is started, modify the `configuration/dask.py` file. 

The workflow is highly modular, making it straightforward to adapt to different datasets or analysis needs. Once the ND2 files have been converted to OME-Zarr, the subsequent steps can be performed independently allowing you to skip or repeat steps as required for your analysis.


## 3. Nucleus segmentation using Cellpose
This notebook focuses on **nucleus segmentation using Cellposev4**. It can be seen as an alternative for StarDist. 

Please ensure that the previous steps (ND2 to OME-Zarr conversion and colony segmentation) have been completed before running this notebook.

In [ ]:
# Initialize Dask cluster and StarDist model for nucleus segmentation
from configuration.dask import (
    start_dask_cluster,
)  # Custom function to start a Dask cluster
from dask.distributed import progress  # For progress visualization of Dask tasks

from trackastra.model import Trackastra
import tenserflow as tf

if tf.cuda.is_available():
    model = Trackastra.from_pretrained("general_2d", device="cuda")
else:
    model = Trackastra.from_pretrained("general_2d")


from configuration.settings import (
    get_output_path,
    get_fovs,
)  # Functions to get output path and list of FOVs
import configuration.settings as settings  # For additional settings (e.g., normalization axis)



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.11.13 
torch version:  	2.8.0+cu128! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 




In [ ]:
# Main tracking workflow: load data, segment nuclei, and save results
import ome_zarr.scale  # For scaling OME-Zarr data
import ome_zarr.reader as ozr  # For reading OME-Zarr data
import ome_zarr.io as ozi  # For OME-Zarr I/O operations
import ome_zarr.writer as ozw  # For writing label data to OME-Zarr
import dask.array as da  # For handling large arrays with Dask
import numpy as np  # For numerical operations
import zarr  # For Zarr storage
import os  # For file path operations
import tqdm  # For progress bars

from trackastra.tracking import graph_to_ctc, graph_to_napari_tracks
import pickle
import gzip


# Utility function to save label arrays to OME-Zarr format
def save_labels(label, label_name, root):
    # Remove existing label if present to avoid duplicates
    if "labels" in root:
        if label_name in root.labels.attrs["labels"]:
            del root["labels"][label_name]
            current_labels = root.labels.attrs["labels"]
            new_labels = [lbl for lbl in current_labels if lbl != label_name]
            root.labels.attrs["labels"] = new_labels
        try:
            del root["labels"][label_name]
        except:
            pass

    Y_dim = root["0"].shape[-2]
    X_dim = root["0"].shape[-1]
    # Write the label array to the OME-Zarr group
    return ozw.write_labels(
        labels=label,
        group=root,
        name=label_name,
        axes="tyx",
        scaler=ome_zarr.scale.Scaler(max_layer=1),
        chunks=(1, Y_dim, X_dim),
        storage_options={
            "compressor": zarr.storage.Blosc(cname="zstd", clevel=5),
        },
        metadata={"is_grayscale_label": False},
    )


# Function to process a single field of view (FOV)
def process_fov(fov):
    dest = os.path.join(get_output_path(), fov)  # Path to OME-Zarr data
    store = ozi.parse_url(dest, mode="a").store
    root = zarr.group(store=store)
    X_dim = root["0"].shape[-1]
    Y_dim = root["0"].shape[-2]
    nodes = list(ozr.Reader(ozi.parse_url(dest, mode="r"))())

    # Try to load colony mask, otherwise use all-ones mask
    try:
        i_colony = nodes[1].zarr.root_attrs["labels"].index("colony")
        colony = nodes[i_colony + 2].data[0]
    except ValueError:
        colony = da.ones((nodes[0].data.shape[0], Y_dim, X_dim), dtype=bool)

    raw = nodes[0].data[0]
    if "channel_names" in nodes[0].metadata:
        # Get H2B channel index from metadata
        H2B_channel = nodes[0].metadata["channel_names"].index("H2B")
    elif "channel" in nodes[0].metadata:
        # Get H2B channel index from metadata
        H2B_channel = nodes[0].metadata["name"].index("H2B")
    raw = raw[:, H2B_channel, :, :]  # Select H2B channel

    if "nucleus" in nodes[1].zarr.root_attrs["labels"]:
        i_nuc = nodes[1].zarr.root_attrs["labels"].index("nucleus")
        nuc_labels = nodes[i_nuc + 2].data
    elif "nucleus_cellpose" in nodes[1].zarr.root_attrs["labels"]:
        i_nuc = nodes[1].zarr.root_attrs["labels"].index("nucleus_cellpose")
        nuc_labels = nodes[i_nuc + 2].data
    else:
        assert "nucleus" not in nodes[1].zarr.root_attrs["labels"]

    track_graph = model.track(raw, nuc_labels, mode="ilp")
    with gzip.open("tracked/track_graph.pickle.gz", "wb") as f:
        pickle.dump(track_graph, f)

    # Write to cell tracking challenge format
    ctc_tracks, masks_tracked = graph_to_ctc(
        track_graph,
        nuc_labels,
    )

    ctc_tracks.to_parquet("tracked/ctc_tracks.parquet")
    napari_tracks, napari_tracks_graph, _ = graph_to_napari_tracks(track_graph)

    with gzip.open("tracked/napari_tracks.pickle.gz", "wb") as f:
        pickle.dump(napari_tracks, f)

    with gzip.open("tracked/napari_tracks_graph.pickle.gz", "wb") as f:
        pickle.dump(napari_tracks_graph, f)

    save_labels(masks_tracked, "tracked", root)
    return


# Get list of FOVs to process
fovs = get_fovs()
for fov in tqdm.tqdm(fovs, desc="Processing FOV", total=len(fovs)):
    process_fov(fov)

Processing FOV:   0%|          | 0/1 [00:00<?, ?it/s]no parent found for <ome_zarr.reader.Label object at 0x000001F95BDE6790>: None
no parent found for <ome_zarr.reader.Label object at 0x000001F95BDF5710>: None
no parent found for <ome_zarr.reader.Label object at 0x000001F959F0B910>: None
no parent found for <ome_zarr.reader.Label object at 0x000001F9607C0CD0>: None
no parent found for <ome_zarr.reader.Label object at 0x000001F95BC96390>: None
no parent found for <ome_zarr.reader.Label object at 0x000001F96072B450>: None
no parent found for <ome_zarr.reader.Label object at 0x000001F95BDE42D0>: None


[########################################] | 100% Completed | 10.85 ss


Processing FOV: 100%|██████████| 1/1 [00:11<00:00, 11.67s/it]
